In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [14]:
df = pd.read_csv('heart.csv')

In [15]:
print("Jumlah baris dan kolom:", df.shape)
print("nInfo dataset:")
print(df.info())
print("\nCek nilai kosong:")
print(df.isnull().sum())
print("\n5 data pertama:")
df.head()

Jumlah baris dan kolom: (1025, 14)
nInfo dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1025 entries, 0 to 1024
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1025 non-null   int64  
 1   sex       1025 non-null   int64  
 2   cp        1025 non-null   int64  
 3   trestbps  1025 non-null   int64  
 4   chol      1025 non-null   int64  
 5   fbs       1025 non-null   int64  
 6   restecg   1025 non-null   int64  
 7   thalach   1025 non-null   int64  
 8   exang     1025 non-null   int64  
 9   oldpeak   1025 non-null   float64
 10  slope     1025 non-null   int64  
 11  ca        1025 non-null   int64  
 12  thal      1025 non-null   int64  
 13  target    1025 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 112.2 KB
None

Cek nilai kosong:
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
sl

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,52,1,0,125,212,0,1,168,0,1.0,2,2,3,0
1,53,1,0,140,203,1,0,155,1,3.1,0,0,3,0
2,70,1,0,145,174,0,1,125,1,2.6,0,0,3,0
3,61,1,0,148,203,0,1,161,0,0.0,2,1,3,0
4,62,0,0,138,294,1,1,106,0,1.9,1,3,2,0


In [16]:
print("Distribusi targer:")
print("df['targe].value_counts")
print("\n0 = Tidak sakit jantung")
print("1 = Sakit jantung")

Distribusi targer:
df['targe].value_counts

0 = Tidak sakit jantung
1 = Sakit jantung


In [20]:
X = df.drop(columns=['target'])
y = df['target']

In [21]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [22]:
print("=" * 45)
print("SKENARIO A - Variasi Nilai K (test_size=0.2)")
print("=" * 45)

hasil_k = []
k_list = [3,5, 7, 9, 11]

SKENARIO A - Variasi Nilai K (test_size=0.2)


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [24]:
for k in k_list:
  model = KNeighborsClassifier(n_neighbors=k)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  acc = accuracy_score(y_test, y_pred)
  hasil_k.append({'K': k, 'Akurasi': round(acc * 100, 2)})

In [26]:
df_hasil_k = pd.DataFrame(hasil_k)
print(df_hasil_k.to_string(index=False))
print("\nK terbaik:", df_hasil_k.loc[df_hasil_k['Akurasi'].idxmax(), 'K'],
      "dengan akurasi", df_hasil_k['Akurasi'].max(), "%")

 K  Akurasi
 3    93.66
 5    83.41
 7    83.90
 9    85.37
11    82.44

K terbaik: 3 dengan akurasi 93.66 %


In [29]:
print("=" * 45)
print("SKENARIO B - Variasi Test Size (K=5)")
print("=" * 45)

hasil_ts = []
test_sizes = [0.2, 0.3, 0.4]

SKENARIO B - Variasi Test Size (K=5)


In [32]:
for ts in test_sizes:
  X_train, X_test, y_train, y_test = train_test_split(
      X_scaled, y, test_size=ts, random_state=42
  )
  model = KNeighborsClassifier(n_neighbors=5)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  acc = accuracy_score(y_test, y_pred)
  hasil_ts.append({
      'Test Size': ts,
      'Train Size': round(1 - ts, 1),
      'Akurasi': round(acc *100, 2)
  })

In [34]:
df_hasil_ts = pd.DataFrame(hasil_ts)
print(df_hasil_ts.to_string(index=False))
print("\nTest size terbaik:", df_hasil_ts.loc[df_hasil_ts['Akurasi'].idxmax(), 'Test Size'],
      "dengan akurasi", df_hasil_ts['Akurasi'].max(), "%")

 Test Size  Train Size  Akurasi
       0.2         0.8    83.41
       0.3         0.7    85.71
       0.4         0.6    85.85

Test size terbaik: 0.4 dengan akurasi 85.85 %


In [35]:
best_k = df_hasil_k.loc[df_hasil_k['Akurasi'].idxmax(), 'K']
best_ts = df_hasil_ts.loc[df_hasil_ts['Akurasi'].idxmax(), 'Test Size']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=best_ts, random_state=42
)

In [38]:
model_terbaik = KNeighborsClassifier(n_neighbors=best_k)
model_terbaik.fit(X_train, y_train)
y_pred = model_terbaik.predict(X_test)

In [40]:
print("=" * 45)
print(f"MODEL TERBAIK - K={best_k}, test_size={best_ts}")
print("=" * 45)
print(f"Akurasi: {round(accuracy_score(y_test, y_pred) * 100, 2)} %")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

MODEL TERBAIK - K=3, test_size=0.4
Akurasi: 92.68 %

Confusion Matrix:
[[189  13]
 [ 17 191]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.94      0.93       202
           1       0.94      0.92      0.93       208

    accuracy                           0.93       410
   macro avg       0.93      0.93      0.93       410
weighted avg       0.93      0.93      0.93       410



In [41]:
print("""
Analisis hasil KNN - Heart Disease Dataset
Dataset : Heart Disease (Kaggle) - 1025 baris, 13 fitur
Target : 0 = tidak sakit jantung, 1 = sakit jantung

Skenario A (variasi K):
- Semakin kecil K, model lebih sensitif tapi rawan overfitting
- Semakin besar K, model lebih stabil tapi bisa underfitting
- K terbaik ditemukan pada K={best_k}

Skenario B (variasi test_size):
- test_size kecil (0.2) = data latih lebih banyak, model lebih terlatih
- test_size besar (0.4) = data uji lebih banyak, evaluasi lebih luas
- test_size terbaik pada {best_ts}

Kesimpulan:
- KNN mampu mengklasifikasi penyakit jantung dengan baik
- Normalisasi data (StandardScaler) penting agar jarak antar
fitur tidak bisa karena perbedaan skala nilai
- Pemilihan K yang tepat sangag memengaruhi hasil akurasi
""".format(best_k=best_k, best_ts=best_ts))


Analisis hasil KNN - Heart Disease Dataset
Dataset : Heart Disease (Kaggle) - 1025 baris, 13 fitur
Target : 0 = tidak sakit jantung, 1 = sakit jantung

Skenario A (variasi K):
- Semakin kecil K, model lebih sensitif tapi rawan overfitting
- Semakin besar K, model lebih stabil tapi bisa underfitting
- K terbaik ditemukan pada K=3

Skenario B (variasi test_size):
- test_size kecil (0.2) = data latih lebih banyak, model lebih terlatih
- test_size besar (0.4) = data uji lebih banyak, evaluasi lebih luas
- test_size terbaik pada 0.4

Kesimpulan: 
- KNN mampu mengklasifikasi penyakit jantung dengan baik
- Normalisasi data (StandardScaler) penting agar jarak antar
fitur tidak bisa karena perbedaan skala nilai
- Pemilihan K yang tepat sangag memengaruhi hasil akurasi

